In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_gv = '/content/drive/MyDrive/tmp/'

In [ ]:
import sys

sys.path.append(path_gv)


In [ ]:
import os


import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from dataset import *
from model import *
from trainer import Trainer

import torch
torch.manual_seed(333)

In [ ]:
#  PATH = "../"
MAX_LEN = 256
BATCH_SIZE = 128

# Loading data

In [ ]:
train_data = pd.read_csv(os.path.join(path_gv, "train.csv"))
test_data = pd.read_csv(os.path.join(path_gv, "test.csv"))

train_data.head()

# Label encoding

In [ ]:
le = LabelEncoder()

train_data.rate = le.fit_transform(train_data.rate)
train_data.head()

# Train Test split

In [ ]:
train_split, val_split = train_test_split(train_data, test_size=0.85, random_state=333)

# Loading tokenizer from pretrained

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "cointegrated/rubert-tiny2", truncation=True, do_lower_case=True)

# Creating datasets and dataloaders

In [ ]:
test_data['rate'] = 0

In [ ]:
train_dataset = FiveDataset(train_split, tokenizer, MAX_LEN)
val_dataset = FiveDataset(val_split, tokenizer, MAX_LEN)
test_dataset = FiveDataset(test_data, tokenizer, MAX_LEN)

In [ ]:
train_params = {"batch_size": BATCH_SIZE,
                "shuffle": True,
                "num_workers": 0
                }

test_params = {"batch_size": BATCH_SIZE,
               "shuffle": False,
               "num_workers": 0
               }

train_dataloader = DataLoader(train_dataset, **train_params)
val_dataloader = DataLoader(val_dataset, **test_params)
test_dataloader = DataLoader(test_dataset, **test_params)

# Loading pretrained model from Huggingface

In [ ]:
config = {
    "num_classes": 5,
    "dropout_rate": 0.3
}
model = ModelForClassification(
    "cointegrated/rubert-tiny2",
    config=config
)

# Creating Trainer object and fitting the model

In [ ]:
trainer_config = {
    "lr": 2e-5,
    "n_epochs": 10,
    "weight_decay": 1e-6,
    "batch_size": BATCH_SIZE,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 333,
}
t = Trainer(trainer_config)

In [ ]:
t.fit(
    model,
    train_dataloader,
    val_dataloader
)

# Save model

In [ ]:
t.save("baseline_model.ckpt")

In [ ]:
# path_full = path_gv + "baseline_model.ckpt"
# t.save(path_full)

# Load pretrained Model

In [ ]:
t = Trainer.load("baseline_model.ckpt")

# Get testset predictions


In [ ]:
predictions = t.predict(test_dataloader)

# Create submission


In [ ]:
sample_submission = pd.read_csv(os.path.join(path_gv, "sample_submission.csv"))
sample_submission["rate"] = predictions
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

In [ ]:
# sample_submission.to_csv(path_gv + "submission.csv", index=False)

In [ ]:
sample_submission.to_csv(path_gv + "submission_007.csv", index=False)